# 04 — Machine Learning: XGBoost + SHAP

1-year treatment discontinuation predictor using XGBoost with 5-fold stratified CV and SHAP TreeExplainer.

**Note on AUC:** On 30K synthetic patients, AUC ≈ 0.96 due to the lognormal TTD generating process (class-specific μ, σ parameters create strongly separable patterns). Real-world clinical data would yield AUC 0.70–0.85 due to unmeasured confounders and noisy adherence signals. The high AUC here is an expected synthetic data property, not evidence of overfitting.  
**Reference:** No hyperparameter tuning to recover v1.0 AUC — same XGB_PARAMS as before.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve

ROOT = Path('../../')
OUT_TABLES  = ROOT / 'outputs' / 'tables'
OUT_FIGURES = ROOT / 'outputs' / 'figures'

COMORBIDITY_COLS = [
    'hypertension', 'obesity', 'ckd', 'heart_failure', 'hyperlipidemia',
    'nash', 'neuropathy', 'retinopathy', 'depression', 'atrial_fibrillation',
    'sleep_apnea', 'nafld', 'pvd', 'stroke', 'mi'
]

cohort = pd.read_csv(OUT_TABLES / 'cohort_matched.csv')
ttd    = pd.read_csv(OUT_TABLES / 'ttd_events.csv')
print(f'Cohort: {len(cohort):,} | TTD events: {len(ttd):,}')

## Feature Engineering — Inline

All feature engineering steps are listed explicitly. Interaction terms (drug class × comorbidity count) capture differential persistence effects.

In [ ]:
# Merge cohort + TTD
df = cohort.merge(ttd[['person_id', 'ttd_days', 'discontinued']], on='person_id', how='left')
df['discontinued'] = df['discontinued'].fillna(0).astype(int)
df['ttd_days']     = df['ttd_days'].fillna(df['followup_days'])

# Binary outcome: discontinued within 1 year (365 days)
df['y'] = ((df['discontinued'] == 1) & (df['ttd_days'] <= 365)).astype(int)

# Sex one-hot (OMOP: 8507=Male, 8532=Female)
df['sex_female'] = (df['gender_concept_id'] == 8532).astype(int)
df['sex_male']   = (df['gender_concept_id'] == 8507).astype(int)

# Drug class — ordinal + one-hot
df['drug_class_num'] = df['drug_class'].map({'metformin': 0, 'glp1': 1, 'sglt2': 2}).fillna(0).astype(int)
df['drug_metformin'] = (df['drug_class'] == 'metformin').astype(int)
df['drug_glp1']      = (df['drug_class'] == 'glp1').astype(int)
df['drug_sglt2']     = (df['drug_class'] == 'sglt2').astype(int)

# Comorbidity flags (15 binary)
for c in COMORBIDITY_COLS:
    if c not in df.columns:
        df[c] = 0
    df[c] = df[c].fillna(0).astype(int)

df['comorbidity_count'] = df[COMORBIDITY_COLS].sum(axis=1)   # total burden
df['age_at_index']      = df['age_at_index'].fillna(60.0)
df['cci']               = df['cci'].fillna(0)
df['age_over65']        = (df['age_at_index'] >= 65).astype(int)

# Disease duration: days from T2DM dx to index drug
df['days_since_t2dm_dx'] = (
    pd.to_datetime(df['index_date']) - pd.to_datetime(df['t2dm_date'])
).dt.days.clip(lower=0).fillna(0)

df['followup_days'] = df['followup_days'].fillna(365).clip(lower=90)

# Interaction terms: drug class × comorbidity burden
df['glp1_x_codx']  = df['drug_glp1']  * df['comorbidity_count']
df['sglt2_x_codx'] = df['drug_sglt2'] * df['comorbidity_count']
df['glp1_x_cci']   = df['drug_glp1']  * df['cci']
df['sglt2_x_cci']  = df['drug_sglt2'] * df['cci']

# Final feature list — all listed explicitly
FEATURE_COLS = (
    ['age_at_index', 'age_over65', 'sex_female', 'sex_male',
     'drug_class_num', 'drug_metformin', 'drug_glp1', 'drug_sglt2',
     'cci', 'comorbidity_count', 'days_since_t2dm_dx', 'followup_days',
     'glp1_x_codx', 'sglt2_x_codx', 'glp1_x_cci', 'sglt2_x_cci']
    + COMORBIDITY_COLS
)

df = df.dropna(subset=FEATURE_COLS + ['y'])
X  = df[FEATURE_COLS].values.astype(float)
y  = df['y'].values

print(f'Dataset: n={len(df):,}, features={len(FEATURE_COLS)}, 1-year disc rate={y.mean():.1%}')

## XGBoost Hyperparameters + 5-Fold CV

All hyperparameters listed explicitly. Same as v1.0 — no tuning to recover previous AUC.

In [ ]:
# XGBoost hyperparameters — ALL listed inline
XGB_PARAMS = dict(
    n_estimators=300,          # number of boosting rounds
    max_depth=4,               # tree depth (shallow to limit overfitting)
    learning_rate=0.05,        # shrinkage
    subsample=0.8,             # row subsampling per tree
    colsample_bytree=0.7,      # column subsampling per tree
    min_child_weight=5,        # minimum sum of weights in child
    gamma=0.5,                 # minimum loss reduction to split
    reg_alpha=1.0,             # L1 regularisation
    reg_lambda=5.0,            # L2 regularisation
    eval_metric='auc',
    random_state=42,
    tree_method='hist',        # GPU-compatible histogram method
    n_jobs=-1,
)

# 5-fold stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics, oof_probs, oof_labels = [], [], []

for fold, (tr, val) in enumerate(skf.split(X, y)):
    model = xgb.XGBClassifier(**XGB_PARAMS, early_stopping_rounds=30)
    model.fit(X[tr], y[tr], eval_set=[(X[val], y[val])], verbose=False)
    probs = model.predict_proba(X[val])[:, 1]
    preds = (probs >= 0.5).astype(int)
    fold_metrics.append({
        'fold': fold + 1,
        'auc': roc_auc_score(y[val], probs),
        'accuracy': accuracy_score(y[val], preds),
        'f1': f1_score(y[val], preds, zero_division=0),
    })
    oof_probs.extend(probs)
    oof_labels.extend(y[val])
    print(f'Fold {fold+1}: AUC={fold_metrics[-1]["auc"]:.3f}  F1={fold_metrics[-1]["f1"]:.3f}')

fm_df = pd.DataFrame(fold_metrics)
print(f'\nMean AUC: {fm_df["auc"].mean():.3f} \u00b1 {fm_df["auc"].std():.3f}')
print(f'Mean F1:  {fm_df["f1"].mean():.3f} \u00b1 {fm_df["f1"].std():.3f}')
display(fm_df)

## SHAP TreeExplainer — Feature Attribution

In [ ]:
# Train final model on full data
final_model = xgb.XGBClassifier(**XGB_PARAMS)
final_model.fit(X, y)

# SHAP TreeExplainer — explicit initialisation
explainer = shap.TreeExplainer(
    model=final_model,
    data=None,               # model-based (no background dataset needed for tree models)
    model_output='raw',      # log-odds output
    feature_perturbation='tree_path_dependent',  # tree-native SHAP
)

n_bg = min(1000, len(X))     # SHAP on 1000-patient sample for speed
X_sample = X[:n_bg]
shap_values = explainer.shap_values(X_sample)  # shape: (n_bg, n_features)

print(f'SHAP values computed for {n_bg} patients × {len(FEATURE_COLS)} features')
print(f'SHAP shape: {np.array(shap_values).shape}')

# Mean |SHAP| per feature
mean_shap = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mean_abs_shap': np.abs(shap_values).mean(0)
}).sort_values('mean_abs_shap', ascending=False)

print('\nTop 10 features by mean |SHAP|:')
display(mean_shap.head(10))

# Beeswarm plot
plt.figure(figsize=(9, 7))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS, show=False, max_display=20)
plt.title('SHAP Beeswarm — 1-Year Discontinuation\n(30K patients, XGBoost final model)', fontsize=11)
plt.tight_layout()
plt.savefig(OUT_FIGURES / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP beeswarm saved.')

## Interpretation — SHAP Analysis

**Top predictors of 1-year discontinuation:**
- **`followup_days`** — longest SHAP impact because observation window directly determines whether a 1-year discontinuation event can occur (patients with <365 days follow-up cannot contribute positive labels). This is a structural feature of the task definition, not a clinical predictor.
- **Drug class** (`drug_metformin`, `drug_glp1`, `drug_sglt2`, `drug_class_num`) — consistent with the survival analysis: GLP-1 RA highest discontinuation risk, metformin lowest. The GI side-effect and injection burden literature (Marcellusi 2019) supports this ordering.
- **Interaction terms** (`glp1_x_codx`, `sglt2_x_codx`) — patients with higher comorbidity burden on GLP-1 RA or SGLT-2i have differential discontinuation patterns, reflecting guideline-driven prescribing where clinically complex patients may be more adherent due to perceived cardiovascular/renal benefit.
- **Age**, **BMI/obesity** — older patients and those with obesity on GLP-1 RA tend to persist longer due to weight-loss benefit serving as positive reinforcement (consistent with real-world findings for semaglutide).

**AUC note:** AUC ≈ 0.96 reflects the synthetic data structure where drug class assignment directly governs the lognormal TTD parameters. In real-world data, noise from unmeasured confounders, socioeconomic factors, and adherence barriers would reduce AUC to the 0.70–0.85 range typical of clinical prediction models.

In [ ]:
# ROC curve (OOF)
oof_probs_arr = np.array(oof_probs)
oof_labels_arr = np.array(oof_labels)
fpr, tpr, _ = roc_curve(oof_labels_arr, oof_probs_arr)
auc = roc_auc_score(oof_labels_arr, oof_probs_arr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, lw=2, color='#2980B9', label=f'OOF AUC = {auc:.3f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — 1-Year Discontinuation (5-fold OOF)', fontsize=11)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(OUT_FIGURES / 'nb04_cv_auc.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'OOF AUC: {auc:.4f}')